# 05. Dimensionality Reduction

Objetivo: Utilizar UMAP para extraer manifolds no-lineales, priorizando la estructura global (`n_neighbors` alto), seguido de un repintado de escala base vía RobustScaler.

In [ ]:
import pandas as pd
import numpy as np
import umap.umap_ as umap
from sklearn.preprocessing import RobustScaler
import logging

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

def reduce_dimensions_umap(df: pd.DataFrame, feature_cols: list, n_neighbors: int = 50, n_components: int = 3) -> pd.DataFrame:
    """
    Applies UMAP to condense feature space while preserving high-dimensional global geometry,
    scaling the resulting vectors geometrically to eliminate magnitude biases.
    """
    logger.info("Starting UMAP dimensionality reduction.")
    df_out = df.copy()
    
    valid_features = [f for f in feature_cols if f in df_out.columns]
    
    if len(valid_features) < 2 or df_out.empty:
        logger.warning("Insufficient features or observations for UMAP.")
        return df_out
        
    # Standardize data defensively before UMAP
    baseline_matrix = df_out[valid_features].fillna(0).values
    
    # Robust UMAP structural settings
    reducer = umap.UMAP(
        n_neighbors=n_neighbors, 
        n_components=n_components, 
        min_dist=0.1, 
        random_state=42
    )
    
    logger.info("Fitting UMAP model (this might take a while)...")
    embeddings = reducer.fit_transform(baseline_matrix)
    
    logger.info("Scaling embedded representations via RobustScaler.")
    scaler = RobustScaler()
    scaled_embeddings = scaler.fit_transform(embeddings)
    
    # Appending components to the dataframe
    for i in range(n_components):
        df_out[f'umap_comp_{i+1}'] = scaled_embeddings[:, i]
        
    return df_out
